### eleconomista webscraper
##### author: Simone Carsey

##### Setup

In [16]:
import requests
from requests.exceptions import ConnectionError 
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
from collections import Counter
import time
import json

In [2]:
with open("headers.json", "r") as f:
    headers_dict_1 = json.load(f)

In [3]:
with open("headers_2.json", "r") as f:
    headers_dict_2 = json.load(f)

In [64]:
csv_entries = pd.read_csv('electrical_entries_2.csv', encoding ='latin1', names = ['company names'])
csv_entries['names_sans_accents'] = csv_entries['company names'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')

In [65]:
def url_search_term(business_name: str):
    business_name = business_name.replace(",", "")
    business_name = business_name.replace(".", "")
    business_name = business_name.replace("(", "")
    business_name = business_name.replace(")", "")
    business_name = business_name.replace(' & ', " ")
    url_st = business_name.upper().replace(' ', '-')
    return url_st

##### Test cells

In [ ]:
later_entries = csv_entries.iloc[1198:,:]
later_entries['company names_sans_accents'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')

1198                                   CASA MESTRET, S.L.
1199                                 CESAR GONZALEZ, S.L.
1200    PREFABRICADOS Y POSTES DE HORMIGON, S.A. (PREP...
1201              TRANSPORTES Y EXCAVACIONES RIBERA, S.A.
1202              EXCAVACIONES Y TRANSPORTES CEREZO, S.A.
                              ...                        
2414                          FERRETERIA ALMATRICHE, S.L.
2415                              ELECTRO SHOW ROOM, S.L.
2416                   GRUPOTEC SERVICIOS AVANZADOS, S.A.
2417                                     GUADACORTE, S.A.
2418                                     TRANSGRUMA, S.A.
Name: company names, Length: 1221, dtype: object

In [77]:
company = csv_entries.iloc[2400]['company names']

mini_header_dict = {
            'User-Agent' : headers_dict_1['48']
        }

baseurl = 'https://empresite.eleconomista.es/Actividad/'
header_index = 5

search_term = url_search_term(company)
search_url = baseurl + search_term + '/'
print(search_url)
r = requests.get(search_url,headers=mini_header_dict, timeout=30)
soup = BeautifulSoup(r.content)
url_results = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))
soup

https://empresite.eleconomista.es/Actividad/JESÚS-AGUILAR/


<!DOCTYPE HTML   >
<html class="h-full" lang="es"><head><title>Directorio de empresas y profesionales de España - Empresite</title><script nonce="Ahquoowai8">window.dataLayer = [{'canal': 'empresite','tipoWeb': 'estatico','propietario': 'externo'}];</script>
<script nonce="Ahquoowai8">(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({'gtm.start':new Date().getTime(),event:'gtm.js'});var f=d.getElementsByTagName(s)[0],j=d.createElement(s),dl=l!='dataLayer'?'&l='+l:'';
j.async=true;j.src='//www.googletagmanager.com/gtm.js?id='+i+dl;f.parentNode.insertBefore(j,f);})(window,document,'script','dataLayer','GTM-WW4JDM');</script>
<meta content="width=device-width, initial-scale=1, user-scalable=1" name="viewport"/>
<meta content="https://empresite.eleconomista.es/servlet/app/prod/PAGINA_NO_ENCONTRADA" property="og:url"/>
<meta content="Empresite España es el mayor directorio de empresas, profesionales, negocios y lugares de España. Buscador gratuito para potenciar su marca, productos, trabajos y 

##### Functions + cells to run them

In [ ]:
def business_search_rotate(csv_entries: pd.DataFrame, n: float, header_dict: dict):
    """
    This function searches the baseurl site for the business names given in the DataFrame csv_entries.
    
    Parameters
    ----------
    """
    link_list = []
    baseurl = 'https://empresite.eleconomista.es/Actividad/'
    header_index = 5

    for i, company in enumerate(tqdm(list(csv_entries['names_sans_accents']))):
        search_term = url_search_term(company)
        search_url = baseurl + search_term + '/'
        mini_header_dict = {
            'User-Agent' : header_dict[str(header_index)]
        }
        try:
            r = requests.get(search_url,headers=mini_header_dict, timeout=30)
            soup = BeautifulSoup(r.content)
            # print(company)
            # print(str(r.status_code))
            try:
                if str(r.status_code)== '200':
                    # first_result = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))[0].text
                    # first_result = first_result.capitalize().split(' ')[0]
                    # company_name_start = company.split(' ')[0]
                    
                    # if first_result.split(' ')[0]==company.split(' ')[0]:
                    url_results = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))[0]['href']
                    link_list.append(url_results)
                elif str(r.status_code)== '429':
                    header_index = header_index + 1
                    try:
                        r = requests.get(search_url,headers=mini_header_dict, timeout=30)
                        soup = BeautifulSoup(r.content)
                        url_results = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))[0]['href']
                        link_list.append(url_results)
                    except:
                        link_list.append('reset but result issue')
                else:
                    link_list.append('possible 404 not found')
            except IndexError:
                link_list.append('no results found')
            time.sleep(n)
        except:
            link_list.append('requests issue')

    print(f'The cooldown this time was {n} seconds.')
    print(f'Last index in dict used was {header_index}')
    return link_list

In [67]:
link_list = business_search_rotate(csv_entries, n = 2, header_dict=headers_dict_1)

100%|██████████| 2861/2861 [1:36:18<00:00,  2.02s/it]  

The cooldown this time was 2 seconds.


In [68]:
Counter(link_list)

Counter({'requests issue': 836,
         'possible 404 not found': 48,
         'https://empresite.eleconomista.es/CB.html': 7,
         'reset but result issue': 6,
         'https://empresite.eleconomista.es/ELECTRODOMESTICOS-CASABLANCA.html': 6,
         'https://empresite.eleconomista.es/SCP.html': 5,
         'https://empresite.eleconomista.es/MUEBLES-ELECTRODOMESTICOS-RAUL.html': 4,
         'https://empresite.eleconomista.es/JOSE-CARLOS-SANCHEZ-NAVARRO-MILLAN.html': 3,
         'https://empresite.eleconomista.es/ELECTROGEN-DIVISION.html': 3,
         'https://empresite.eleconomista.es/JIMENEZ-GARCIA-FRANCISCO-JOSE.html': 2,
         'https://empresite.eleconomista.es/ANTONIO-SANCHEZ-LARA.html': 2,
         'https://empresite.eleconomista.es/LARA-RODRIGO-SERVICIOS.html': 2,
         'https://empresite.eleconomista.es/ELECTRODOMESTICOS-BERNABE-JURADO.html': 2,
         'https://empresite.eleconomista.es/PANASONIC-ESPANA-SUCURSAL-MARKETING-EUROPE-GMBH.html': 2,
         'https://em

In [10]:
def cleanup_found_links_rotate(df: pd.DataFrame, link_list: list) -> None:
    """
    This function adds urls to company listings that were found and removes rows of companies which were not found listed.

    Parameters
    ----------
    'df' : pd.DataFrame
        DataFrame of companies from csv import.
    'link_list' : list of links found from 'business_search' 
    """
    df = df.copy()
    df.reset_index(drop=True,inplace=True)
    df['found_links'] = pd.DataFrame(link_list, columns=['found_links'])
    excluded_rows = df[(df['found_links']=='possible 404 not found') | (df['found_links']=='reset but result issue') | (df['found_links']=='429 issue') | (df['found_links']=='requests issue')].index
    df.drop(index=excluded_rows, inplace=True)
    df = df.drop_duplicates(subset=['found_links'], keep=False) #mutiplicities of url found are generally because of search issues
    df = df.dropna(subset='found_links')
    df.reset_index(drop=True,inplace=True)

    return df

In [78]:
entries_infoadded = cleanup_found_links_rotate(csv_entries, link_list)

In [ ]:
def get_contact_info_rotate(df: pd.DataFrame, n: int, header_dict: dict):
    """
    This company scrapes info of companies for which links were found.

    Parameters
    ----------
    'df' : pd.DataFrame
    """
    phonenumbers_found = []
    urls_found = []
    emails_found = []
    header_index = 5

    for i, link in enumerate(tqdm(df['found_links'])):
        mini_header_dict = {
            'User-Agent' : header_dict[str(header_index)]
        }
        try:
            r = requests.get(link,headers=mini_header_dict, timeout=20)
            if r.status_code == 429:
                header_index = header_index + 1
                r = requests.get(link,headers=mini_header_dict, timeout=20)
            if (r.status_code != 200) & (r.status_code != 429):
                print(str(r.status_code))
            soup = BeautifulSoup(r.content)
            try:
                found_url = soup.find_all('a', href=True, target = '_blank', class_ = lambda value: value and value.startswith('text-bodytext-m text-secondary-800 underline'))[0]['href']
            except:
                found_url='not found'
            urls_found.append(found_url)
            try:
                found_email = soup.find_all('a', href=True, target = '_blank', class_ = lambda value: value and value.startswith('text-bodytext-m text-secondary-800 underline email'))[0]['href'].split('?')[0]
            except:
                found_email = 'not found'
            emails_found.append(found_email)
            try:
                found_phone = soup.find_all('span', class_=lambda value: value and value.startswith('text-bodytext-m text-neutrals-700 md:block hidden'))[0].text
            except:
                found_phone = 'not found'
            phonenumbers_found.append(found_phone)
            time.sleep(n)
        except ConnectionError:
            found_url='connection error not found'
            urls_found.append(found_url)
            found_email = 'connection error not found'
            emails_found.append(found_email)
            found_phone = 'connection error not found'
            phonenumbers_found.append(found_phone)

    info_dict = {'phones': phonenumbers_found, 'urls': urls_found, 'emails': emails_found}
    print(f'The cooldown this time was {n} seconds.')
    print(f'Last index in dict used was {header_index}')
    return info_dict

In [ ]:
contact_info = get_contact_info_rotate(entries_infoadded, n=2, header_dict=headers_dict_2)

  0%|          | 0/1907 [00:00<?, ?it/s]

 27%|██▋       | 509/1907 [21:10<59:40,  2.56s/it]  

In [ ]:
def add_found_info(df: pd.DataFrame, contact_info_dict: dict)-> None:
    """
    compiles phone numbers, emails, urls into df and fixes some formatting issues.

    Parameters
    ----------
    """
    phone_list = contact_info_dict['phones']
    url_list = contact_info_dict['urls']
    email_list = contact_info_dict['emails']
    
    df['phones'] = pd.DataFrame(phone_list, columns=['phones'])
    df['urls'] = pd.DataFrame(url_list, columns=['urls'])
    df['emails'] = pd.DataFrame(email_list, columns=['emails'])

    # Cleaning Results:
    df['urls'] = df['urls'].apply(lambda x: 'not found' if x.startswith('mailto')==True else x)
    df['urls'] = df['urls'].apply(lambda x: 'not found' if x.startswith('https://ranking-empresas.eleconomista.es/')==True else x)

    # Format corrections:
    df['urls'] = df['urls'].apply(lambda x: x[2:] if x.startswith('//')==True else x)
    df['emails'] = df['emails'].apply(lambda x: x.split(':')[1] if x.startswith('mailto')==True else x)
    df.drop(columns=['found_links','names_sans_accents'],inplace=True)

    # Removing companies w no info:
    # no_info_indices = df.loc[(df['emails']=='not found') & (df['phones']=='not found') & (df['urls']=='not found')].index
    # df.drop(index = no_info_indices, inplace=True)
    df.reset_index(drop=True, inplace=True)
    return None

In [40]:
add_found_info(df=entries_infoadded, contact_info_dict=contact_info)

In [ ]:
entries_infoadded.to_csv('electrical_revised_search_eleconomista.csv')